In [1]:
!apt-get install -y ocl-icd-opencl-dev opencl-headers clinfo
!clinfo | head -20

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
clinfo is already the newest version (3.0.21.02.21-1).
ocl-icd-opencl-dev is already the newest version (2.2.14-3).
ocl-icd-opencl-dev set to manually installed.
The following NEW packages will be installed:
  opencl-headers
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 1,754 B of archives.
After this operation, 12.3 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 opencl-headers all 3.0~2022.01.04-1 [1,754 B]
Fetched 1,754 B in 0s (30.9 kB/s)
Selecting previously unselected package opencl-headers.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../opencl-headers_3.0~2022.01.04-1_all.deb ...
Unpacking opencl-headers (3.0~2022.01.04-1) ...
Setting up opencl-headers (3.0~2022.01.04-1) ...
Number of platforms                               1
  Platform Name           

In [2]:
%%writefile kernel.cl

// ── Kernel 1: Asignar cada punto al centroide más cercano ──────────
__kernel void assign_clusters(
    __global const float* points,     // [N x D] aplanado
    __global const float* centroids,  // [K x D] aplanado
    __global       int*   labels,     // [N]
    const int N,
    const int K,
    const int D)
{
    int i = get_global_id(0);
    if (i >= N) return;

    float min_dist = INFINITY;
    int   best_k   = 0;

    for (int k = 0; k < K; k++) {
        float dist = 0.0f;
        for (int d = 0; d < D; d++) {
            float diff = points[i*D + d] - centroids[k*D + d];
            dist += diff * diff;
        }
        if (dist < min_dist) {
            min_dist = dist;
            best_k   = k;
        }
    }
    labels[i] = best_k;
}

// ── Kernel 2: Acumular suma de puntos por cluster ──────────────────
// Cada work-item acumula su punto en el centroide correspondiente.
// Usamos atomic en enteros: multiplicamos float x 1000 para operar como int.
__kernel void accumulate_centroids(
    __global const float* points,   // [N x D]
    __global const int*   labels,   // [N]
    __global       int*   sums,     // [K x D]  (float*1000 como int)
    __global       int*   counts,   // [K]
    const int N,
    const int K,
    const int D)
{
    int i = get_global_id(0);
    if (i >= N) return;

    int k = labels[i];
    atomic_add(&counts[k], 1);
    for (int d = 0; d < D; d++) {
        int val = (int)(points[i*D + d] * 1000.0f);
        atomic_add(&sums[k*D + d], val);
    }
}

Writing kernel.cl


In [4]:
%%writefile main.cpp
#include <CL/cl.h>
#include <iostream>
#include <fstream>
#include <sstream>
#include <vector>
#include <random>
#include <cmath>
#include <chrono>
#include <cassert>
#include <algorithm>
#include <numeric>

using namespace std;
using namespace std::chrono;

// -- Utilidades ----------------------------------------------------

string readFile(const string& path) {
    ifstream f(path);
    if (!f.is_open()) { cerr << "No se pudo abrir: " << path << "\n"; exit(1); }
    return string(istreambuf_iterator<char>(f),
                  istreambuf_iterator<char>());
}

void checkCL(cl_int err, const char* msg) {
    if (err != CL_SUCCESS) {
        cerr << "Error OpenCL [" << err << "] en: " << msg << "\n";
        exit(1);
    }
}

double nowMs() {
    return duration<double, milli>(
        high_resolution_clock::now().time_since_epoch()).count();
}

// -- Listar plataformas y dispositivos disponibles -----------------
void listDevices() {
    cl_uint numPlatforms = 0;
    clGetPlatformIDs(0, nullptr, &numPlatforms);
    cout << "Plataformas encontradas: " << numPlatforms << "\n";

    vector<cl_platform_id> platforms(numPlatforms);
    clGetPlatformIDs(numPlatforms, platforms.data(), nullptr);

    for (cl_uint p = 0; p < numPlatforms; p++) {
        char pname[256];
        clGetPlatformInfo(platforms[p], CL_PLATFORM_NAME, sizeof(pname), pname, nullptr);
        cout << "  Plataforma [" << p << "]: " << pname << "\n";

        cl_uint numDevices = 0;
        clGetDeviceIDs(platforms[p], CL_DEVICE_TYPE_ALL, 0, nullptr, &numDevices);
        vector<cl_device_id> devices(numDevices);
        clGetDeviceIDs(platforms[p], CL_DEVICE_TYPE_ALL, numDevices, devices.data(), nullptr);

        for (cl_uint d = 0; d < numDevices; d++) {
            char dname[256];
            clGetDeviceInfo(devices[d], CL_DEVICE_NAME, sizeof(dname), dname, nullptr);
            cout << "    Dispositivo [" << d << "]: " << dname << "\n";
        }
    }
}

// -- Seleccionar mejor dispositivo (GPU preferido, sino CPU) -------
bool selectDevice(cl_platform_id& platform, cl_device_id& device) {
    cl_uint numPlatforms = 0;
    clGetPlatformIDs(0, nullptr, &numPlatforms);
    if (numPlatforms == 0) { cerr << "No hay plataformas OpenCL\n"; return false; }

    vector<cl_platform_id> platforms(numPlatforms);
    clGetPlatformIDs(numPlatforms, platforms.data(), nullptr);

    // Intentar GPU primero, luego cualquier dispositivo
    for (auto type : {CL_DEVICE_TYPE_GPU, CL_DEVICE_TYPE_ALL}) {
        for (auto& plat : platforms) {
            cl_uint numDevices = 0;
            if (clGetDeviceIDs(plat, type, 0, nullptr, &numDevices) != CL_SUCCESS) continue;
            if (numDevices == 0) continue;

            vector<cl_device_id> devices(numDevices);
            clGetDeviceIDs(plat, type, numDevices, devices.data(), nullptr);
            platform = plat;
            device   = devices[0];
            return true;
        }
    }
    return false;
}

// -- Generador de datos --------------------------------------------
vector<float> generateData(int N, int D, int K, unsigned seed = 42) {
    mt19937 rng(seed);
    normal_distribution<float>       noise(0.0f, 1.5f);
    uniform_real_distribution<float> center(-10.0f, 10.0f);

    vector<float> centers(K * D);
    for (auto& v : centers) v = center(rng);

    vector<float> data(N * D);
    for (int i = 0; i < N; i++) {
        int k = i % K;
        for (int d = 0; d < D; d++)
            data[i*D + d] = centers[k*D + d] + noise(rng);
    }
    for (int i = N-1; i > 0; i--) {
        int j = rng() % (i+1);
        for (int d = 0; d < D; d++)
            swap(data[i*D+d], data[j*D+d]);
    }
    return data;
}

// -- KMeans en CPU (referencia) ------------------------------------
vector<int> kmeansCPU(const vector<float>& X,
                      int N, int D, int K,
                      int maxIter, float tol) {
    mt19937 rng(0);
    vector<float> centroids(K * D);
    vector<int>   labels(N, 0);

    vector<int> idx(N);
    iota(idx.begin(), idx.end(), 0);
    shuffle(idx.begin(), idx.end(), rng);

    for (int k = 0; k < K; k++)
        for (int d = 0; d < D; d++)
            centroids[k*D+d] = X[idx[k]*D+d];

    for (int it = 0; it < maxIter; it++) {
        for (int i = 0; i < N; i++) {
            float best = 1e30f; int bk = 0;
            for (int k = 0; k < K; k++) {
                float dist = 0;
                for (int d = 0; d < D; d++) {
                    float diff = X[i*D+d] - centroids[k*D+d];
                    dist += diff*diff;
                }
                if (dist < best) { best = dist; bk = k; }
            }
            labels[i] = bk;
        }
        vector<float> newC(K*D, 0);
        vector<int>   cnt(K, 0);
        for (int i = 0; i < N; i++) {
            int k = labels[i]; cnt[k]++;
            for (int d = 0; d < D; d++) newC[k*D+d] += X[i*D+d];
        }
        float shift = 0;
        for (int k = 0; k < K; k++) {
            if (cnt[k] == 0) continue;
            for (int d = 0; d < D; d++) {
                float nc = newC[k*D+d] / cnt[k];
                shift += (nc - centroids[k*D+d]) * (nc - centroids[k*D+d]);
                centroids[k*D+d] = nc;
            }
        }
        if (sqrt(shift) < tol) { cout << "CPU convergió en iter " << it+1 << "\n"; break; }
    }
    return labels;
}

// -- KMeans con OpenCL ---------------------------------------------
vector<int> kmeansOpenCL(const vector<float>& X,
                          int N, int D, int K,
                          int maxIter, float tol,
                          double& gpuTimeMs) {
    cl_int err;

    // Seleccionar plataforma y dispositivo automáticamente
    cl_platform_id platform;
    cl_device_id   device;
    if (!selectDevice(platform, device)) {
        cerr << "No se encontró ningún dispositivo OpenCL\n";
        exit(1);
    }

    char devName[256];
    clGetDeviceInfo(device, CL_DEVICE_NAME, sizeof(devName), devName, nullptr);
    cout << "Dispositivo OpenCL: " << devName << "\n";

    cl_context_properties props[] = {
        CL_CONTEXT_PLATFORM, (cl_context_properties)platform, 0
    };
    cl_context ctx = clCreateContext(props, 1, &device, nullptr, nullptr, &err);
    checkCL(err, "clCreateContext");

    // Usar la API moderna clCreateCommandQueueWithProperties
    cl_queue_properties qprops[] = {
        CL_QUEUE_PROPERTIES, CL_QUEUE_PROFILING_ENABLE, 0
    };
    cl_command_queue queue = clCreateCommandQueueWithProperties(ctx, device, qprops, &err);
    checkCL(err, "clCreateCommandQueueWithProperties");

    string src = readFile("kernel.cl");
    const char* srcPtr = src.c_str();
    size_t srcLen = src.size();
    cl_program program = clCreateProgramWithSource(ctx, 1, &srcPtr, &srcLen, &err);
    checkCL(err, "clCreateProgramWithSource");

    err = clBuildProgram(program, 1, &device, "-cl-fast-relaxed-math", nullptr, nullptr);
    if (err != CL_SUCCESS) {
        size_t logSize;
        clGetProgramBuildInfo(program, device, CL_PROGRAM_BUILD_LOG, 0, nullptr, &logSize);
        string log(logSize, ' ');
        clGetProgramBuildInfo(program, device, CL_PROGRAM_BUILD_LOG, logSize, &log[0], nullptr);
        cerr << "Build error:\n" << log << "\n";
        exit(1);
    }

    cl_kernel kernelAssign = clCreateKernel(program, "assign_clusters",      &err); checkCL(err, "kernelAssign");
    cl_kernel kernelAccum  = clCreateKernel(program, "accumulate_centroids", &err); checkCL(err, "kernelAccum");

    mt19937 rng(0);
    vector<int> idx(N);
    iota(idx.begin(), idx.end(), 0);
    shuffle(idx.begin(), idx.end(), rng);

    vector<float> centroids(K * D);
    for (int k = 0; k < K; k++)
        for (int d = 0; d < D; d++)
            centroids[k*D+d] = X[idx[k]*D+d];

    cl_mem buf_X   = clCreateBuffer(ctx, CL_MEM_READ_ONLY  | CL_MEM_COPY_HOST_PTR,
                                    N*D*sizeof(float), (void*)X.data(), &err);        checkCL(err, "buf_X");
    cl_mem buf_C   = clCreateBuffer(ctx, CL_MEM_READ_WRITE | CL_MEM_COPY_HOST_PTR,
                                    K*D*sizeof(float), centroids.data(), &err);        checkCL(err, "buf_C");
    cl_mem buf_L   = clCreateBuffer(ctx, CL_MEM_READ_WRITE,
                                    N*sizeof(int), nullptr, &err);                     checkCL(err, "buf_L");
    cl_mem buf_S   = clCreateBuffer(ctx, CL_MEM_READ_WRITE,
                                    K*D*sizeof(int), nullptr, &err);                   checkCL(err, "buf_S");
    cl_mem buf_Cnt = clCreateBuffer(ctx, CL_MEM_READ_WRITE,
                                    K*sizeof(int), nullptr, &err);                     checkCL(err, "buf_Cnt");

    vector<int> labels(N, 0);
    gpuTimeMs = 0.0;

    for (int it = 0; it < maxIter; it++) {

        // -- Kernel 1: Asignar --------------------------------------
        clSetKernelArg(kernelAssign, 0, sizeof(cl_mem), &buf_X);
        clSetKernelArg(kernelAssign, 1, sizeof(cl_mem), &buf_C);
        clSetKernelArg(kernelAssign, 2, sizeof(cl_mem), &buf_L);
        clSetKernelArg(kernelAssign, 3, sizeof(int),    &N);
        clSetKernelArg(kernelAssign, 4, sizeof(int),    &K);
        clSetKernelArg(kernelAssign, 5, sizeof(int),    &D);

        size_t globalSize = (size_t)N;
        size_t localSize  = 64;
        if (globalSize % localSize != 0)
            globalSize = ((globalSize / localSize) + 1) * localSize;

        cl_event evAssign;
        err = clEnqueueNDRangeKernel(queue, kernelAssign, 1, nullptr,
                                     &globalSize, &localSize, 0, nullptr, &evAssign);
        checkCL(err, "enqueue assign");
        clWaitForEvents(1, &evAssign);

        cl_ulong t_start, t_end;
        clGetEventProfilingInfo(evAssign, CL_PROFILING_COMMAND_START, sizeof(t_start), &t_start, nullptr);
        clGetEventProfilingInfo(evAssign, CL_PROFILING_COMMAND_END,   sizeof(t_end),   &t_end,   nullptr);
        gpuTimeMs += (t_end - t_start) * 1e-6;
        clReleaseEvent(evAssign);

        // -- Kernel 2: Acumular ------------------------------------
        vector<int> zeros_S(K*D, 0), zeros_C(K, 0);
        clEnqueueWriteBuffer(queue, buf_S,   CL_TRUE, 0, K*D*sizeof(int), zeros_S.data(), 0, nullptr, nullptr);
        clEnqueueWriteBuffer(queue, buf_Cnt, CL_TRUE, 0, K*sizeof(int),   zeros_C.data(), 0, nullptr, nullptr);

        clSetKernelArg(kernelAccum, 0, sizeof(cl_mem), &buf_X);
        clSetKernelArg(kernelAccum, 1, sizeof(cl_mem), &buf_L);
        clSetKernelArg(kernelAccum, 2, sizeof(cl_mem), &buf_S);
        clSetKernelArg(kernelAccum, 3, sizeof(cl_mem), &buf_Cnt);
        clSetKernelArg(kernelAccum, 4, sizeof(int),    &N);
        clSetKernelArg(kernelAccum, 5, sizeof(int),    &K);
        clSetKernelArg(kernelAccum, 6, sizeof(int),    &D);

        cl_event evAccum;
        err = clEnqueueNDRangeKernel(queue, kernelAccum, 1, nullptr,
                                     &globalSize, &localSize, 0, nullptr, &evAccum);
        checkCL(err, "enqueue accum");
        clWaitForEvents(1, &evAccum);

        clGetEventProfilingInfo(evAccum, CL_PROFILING_COMMAND_START, sizeof(t_start), &t_start, nullptr);
        clGetEventProfilingInfo(evAccum, CL_PROFILING_COMMAND_END,   sizeof(t_end),   &t_end,   nullptr);
        gpuTimeMs += (t_end - t_start) * 1e-6;
        clReleaseEvent(evAccum);

        // -- Actualizar centroides en host -------------------------
        vector<int> sums(K*D), counts(K);
        clEnqueueReadBuffer(queue, buf_S,   CL_TRUE, 0, K*D*sizeof(int), sums.data(),   0, nullptr, nullptr);
        clEnqueueReadBuffer(queue, buf_Cnt, CL_TRUE, 0, K*sizeof(int),   counts.data(), 0, nullptr, nullptr);

        vector<float> newCentroids(K*D);
        float shift = 0;
        for (int k = 0; k < K; k++) {
            if (counts[k] == 0) {
                for (int d = 0; d < D; d++) newCentroids[k*D+d] = centroids[k*D+d];
                continue;
            }
            for (int d = 0; d < D; d++) {
                float nc = (sums[k*D+d] / 1000.0f) / counts[k];
                shift += (nc - centroids[k*D+d]) * (nc - centroids[k*D+d]);
                newCentroids[k*D+d] = nc;
            }
        }
        centroids = newCentroids;
        clEnqueueWriteBuffer(queue, buf_C, CL_TRUE, 0,
                             K*D*sizeof(float), centroids.data(), 0, nullptr, nullptr);

        if (sqrt(shift) < tol) {
            cout << "GPU convergió en iter " << it+1 << "\n";
            break;
        }
    }

    clEnqueueReadBuffer(queue, buf_L, CL_TRUE, 0, N*sizeof(int), labels.data(), 0, nullptr, nullptr);

    clReleaseMemObject(buf_X); clReleaseMemObject(buf_C);
    clReleaseMemObject(buf_L); clReleaseMemObject(buf_S);
    clReleaseMemObject(buf_Cnt);
    clReleaseKernel(kernelAssign); clReleaseKernel(kernelAccum);
    clReleaseProgram(program);
    clReleaseCommandQueue(queue);
    clReleaseContext(ctx);

    return labels;
}

// -- Inercia -------------------------------------------------------
float inertia(const vector<float>& X,
              const vector<int>&   labels,
              int N, int D, int K) {
    vector<float> c(K*D, 0);
    vector<int>   cnt(K, 0);
    for (int i = 0; i < N; i++) {
        int k = labels[i]; cnt[k]++;
        for (int d = 0; d < D; d++) c[k*D+d] += X[i*D+d];
    }
    for (int k = 0; k < K; k++)
        if (cnt[k] > 0)
            for (int d = 0; d < D; d++) c[k*D+d] /= cnt[k];

    float total = 0;
    for (int i = 0; i < N; i++) {
        int k = labels[i];
        for (int d = 0; d < D; d++) {
            float diff = X[i*D+d] - c[k*D+d];
            total += diff*diff;
        }
    }
    return total;
}

// -- Main ----------------------------------------------------------
int main() {
    const int   N       = 100000;
    const int   D       = 2;
    const int   K       = 5;
    const int   MAXITER = 300;
    const float TOL     = 1e-4f;

    cout << "=== KMeans con OpenCL en C++ ===\n";
    cout << "N=" << N << "  D=" << D << "  K=" << K << "\n\n";

    listDevices();
    cout << "\n";

    auto X = generateData(N, D, K);

    // -- GPU -------------------------------------------------------
    double gpuTimeMs = 0;
    double t0 = nowMs();
    auto labelsGPU = kmeansOpenCL(X, N, D, K, MAXITER, TOL, gpuTimeMs);
    double totalGPU = nowMs() - t0;

    cout << "  Inercia GPU          : " << inertia(X, labelsGPU, N, D, K) << "\n";
    cout << "  Tiempo kernels (GPU) : " << gpuTimeMs << " ms\n";
    cout << "  Tiempo total  (GPU)  : " << totalGPU  << " ms\n\n";

    // -- CPU -------------------------------------------------------
    double t1 = nowMs();
    auto labelsCPU = kmeansCPU(X, N, D, K, MAXITER, TOL);
    double totalCPU = nowMs() - t1;

    cout << "  Inercia CPU          : " << inertia(X, labelsCPU, N, D, K) << "\n";
    cout << "  Tiempo total  (CPU)  : " << totalCPU << " ms\n\n";

    cout << "  Speedup kernels : " << (totalCPU / gpuTimeMs) << "x\n";
    cout << "  Speedup total   : " << (totalCPU / totalGPU)  << "x\n";

    return 0;
}

Writing main.cpp


In [5]:
!g++ -O2 -o kmeans main.cpp -lOpenCL -std=c++17
!./kmeans

In file included from /usr/include/CL/cl.h:20,
                 from main.cpp:1:
/usr/include/CL/cl_version.h:22:104: note: ‘#pragma message: cl_version.h: CL_TARGET_OPENCL_VERSION is not defined. Defaulting to 300 (OpenCL 3.0)’
   22 | #pragma message("cl_version.h: CL_TARGET_OPENCL_VERSION is not defined. Defaulting to 300 (OpenCL 3.0)")
      |                                                                                                        ^
main.cpp: In function ‘bool selectDevice(_cl_platform_id*&, _cl_device_id*&)’:
main.cpp:74:61: error: unable to deduce ‘std::initializer_list<auto>&&’ from ‘{(1 << 2), 4294967295}’
   74 |     for (auto type : {CL_DEVICE_TYPE_GPU, CL_DEVICE_TYPE_ALL}) {
      |                                                             ^
main.cpp:74:61: note:   deduced conflicting types for parameter ‘auto’ (‘int’ and ‘unsigned int’)
/bin/bash: line 1: ./kmeans: No such file or directory
